# Lab 22 — Final Research Capstone

*AI Cybersecurity Masterclass · Lab 22 · AI60secLab*
## Adaptive Cyber Threat Detection Under Shift and Adversarial Conditions

**Core research question:** How does a machine-learning cyber threat detector behave when the training data, input distribution, or decision environment changes, and which defensive intervention best preserves useful detection performance?

### Experimental design

Each condition changes **one factor** relative to a named baseline:

| Condition | What changes | Compared with |
|---|---|---|
| C0 | Baseline model, default 0.5 threshold | (reference) |
| C1 | Threshold chosen on validation data by a fixed rule (max F2) | C0 |
| C2 | C1 evaluated on bounded input stress | C1 |
| C3 | 5% label-poisoned training, same threshold **rule** | C1 |
| C4 | Noise-augmented training, same threshold **rule**, clean and stressed | C1 and C2 |

Rules that keep the comparison fair:
- same data, preprocessing, metrics, and model settings everywhere;
- lock the threshold **rule**, not the number: each retrained model gets its own threshold from the same validation procedure;
- repeat with several seeds and report the spread;
- a delta only counts if it is bigger than the spread between runs;
- save one row per result.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import logistic_pipeline
from src.evaluation import binary_metrics
from src.thresholds import (
    threshold_sweep,
    choose_threshold,
    apply_threshold,
)
from src.drift import domain_classifier_auc
from src.adversarial import (
    available_numeric_attack_features,
    training_bounds,
    bounded_random_perturbation,
    augment_attacks_with_noise,
)
from src.poisoning import random_label_flip

train, test = load_unsw(DATA_DIR)

X_full, y_full = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_full, X_test = align_columns(X_full, X_test)

X_full = drop_identifier_like_columns(X_full)
X_test = drop_identifier_like_columns(X_test)

stress_columns = available_numeric_attack_features(X_full)
bounds = training_bounds(X_full, stress_columns)

SEEDS = [0, 1, 2]

In [ ]:
def fit_with_threshold_rule(X_tr, y_tr, X_va, y_va):
    """Train a model and pick its threshold with the SAME validation rule."""
    model = logistic_pipeline(X_tr)
    model.fit(X_tr, y_tr)
    sweep = threshold_sweep(y_va, model.predict_proba(X_va)[:, 1], beta=2.0)
    threshold, _ = choose_threshold(sweep, objective="f_beta")
    return model, threshold


def result_row(condition, seed, model, X_eval, threshold):
    scores = model.predict_proba(X_eval)[:, 1]
    m = binary_metrics(y_test, apply_threshold(scores, threshold), scores)
    return {
        "condition": condition,
        "seed": seed,
        "threshold": threshold,
        "precision": m.precision,
        "recall": m.recall,
        "f1": m.f1,
        "false_negative_rate": m.false_negative_rate,
        "roc_auc": m.roc_auc,
    }


rows = []
for seed in SEEDS:
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_full, y_full, test_size=0.20, stratify=y_full, random_state=seed,
    )

    baseline, tuned_threshold = fit_with_threshold_rule(X_train, y_train, X_valid, y_valid)
    X_stressed = bounded_random_perturbation(
        X_test, columns=stress_columns, epsilon=0.10, bounds=bounds, random_state=seed,
    )

    rows.append(result_row("C0 baseline, threshold 0.5", seed, baseline, X_test, 0.5))
    rows.append(result_row("C1 validation-selected threshold", seed, baseline, X_test, tuned_threshold))
    rows.append(result_row("C2 C1 under input stress", seed, baseline, X_stressed, tuned_threshold))

    poisoned_y = random_label_flip(y_train, fraction=0.05, random_state=seed)
    poisoned, poisoned_threshold = fit_with_threshold_rule(
        X_train.reset_index(drop=True), poisoned_y, X_valid, y_valid,
    )
    rows.append(result_row("C3 5% label poisoning", seed, poisoned, X_test, poisoned_threshold))

    X_aug, y_aug = augment_attacks_with_noise(
        X_train, y_train, columns=stress_columns, epsilon=0.05, copies=1,
        bounds=bounds, random_state=seed,
    )
    augmented, augmented_threshold = fit_with_threshold_rule(X_aug, y_aug, X_valid, y_valid)
    rows.append(result_row("C4 augmented, clean test", seed, augmented, X_test, augmented_threshold))
    rows.append(result_row("C4 augmented, under stress", seed, augmented, X_stressed, augmented_threshold))

runs = pd.DataFrame(rows)
runs.to_csv(RESULTS_DIR / "22_capstone_runs.csv", index=False)
print("Saved", len(runs), "rows to results/22_capstone_runs.csv")

In [ ]:
domain_auc, _ = domain_classifier_auc(
    X_full,
    X_test,
    sample_per_domain=15000,
    random_state=42,
)

print(
    "Train-vs-test domain classifier AUC:",
    round(domain_auc, 4),
)

In [ ]:
summary = (
    runs.groupby("condition", sort=False)[["threshold", "precision", "recall", "f1", "false_negative_rate"]]
    .agg(["mean", "std"])
)
display(summary.round(4))

## Which deltas count?

Each condition is compared with the condition it changes one factor from. A delta counts only if it is more than twice the largest seed standard deviation of the two conditions being compared.

In [ ]:
comparisons = [
    ("C1 validation-selected threshold", "C0 baseline, threshold 0.5"),
    ("C2 C1 under input stress", "C1 validation-selected threshold"),
    ("C3 5% label poisoning", "C1 validation-selected threshold"),
    ("C4 augmented, clean test", "C1 validation-selected threshold"),
    ("C4 augmented, under stress", "C2 C1 under input stress"),
]

stats = runs.groupby("condition")[["recall", "f1"]].agg(["mean", "std"])
delta_rows = []
for treatment, reference in comparisons:
    for metric in ["recall", "f1"]:
        delta = stats.loc[treatment, (metric, "mean")] - stats.loc[reference, (metric, "mean")]
        spread = max(stats.loc[treatment, (metric, "std")], stats.loc[reference, (metric, "std")])
        delta_rows.append({
            "treatment": treatment,
            "reference": reference,
            "metric": metric,
            "delta": delta,
            "seed_spread": spread,
            "counts": abs(delta) > 2 * spread,
        })

deltas = pd.DataFrame(delta_rows)
display(deltas.round(4))

In [ ]:
print(
    "Independent distribution-shift signal (domain classifier AUC):",
    round(domain_auc, 4),
)
print("Conditions whose deltas exceed the seed spread:")
display(deltas[deltas["counts"]][["treatment", "metric", "delta"]].round(4))

# Final white-paper structure

1. **Abstract**
2. **Research question and falsifiable hypotheses**
3. **Dataset and limitations**
4. **Methods and train/validation/test separation**
5. **Clean results**
6. **Shift, stress, and poisoning results**
7. **Mitigation comparison**
8. **Security interpretation**
9. **Limitations**
10. **Reproducibility and future research**

**Final principle:** do not claim “the model is secure.” State exactly what conditions were tested, what degraded, what improved, and what remains unknown.